In [0]:
# Databricks notebook source

# COMMAND ----------
# Configuración

CATALOG_NAME = "workspace"
SCHEMA_NAME  = "default"

SILVER       = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_retail_sales"
DIM_CUSTOMER = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_customer"
DIM_ITEM     = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_item"
DIM_DATE     = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_date"
FACT_SALES   = f"{CATALOG_NAME}.{SCHEMA_NAME}.fact_sales"

VALID_PAYMENT  = {"cash", "credit card", "digital wallet"}
VALID_LOCATION = {"online", "in-store"}

results = []  # tabla resumen final

def log(name, passed, detail=""):
    status = "✓ OK" if passed else "✗ ERROR"
    results.append((name, status, str(detail)))
    print(f"{status} — {name}" + (f": {detail}" if detail else ""))
    if not passed:
        raise AssertionError(f"Validación crítica fallida: {name} — {detail}")

# COMMAND ----------
# Lectura de tablas

from pyspark.sql import functions as F

silver  = spark.table(SILVER)
fact    = spark.table(FACT_SALES)
d_cust  = spark.table(DIM_CUSTOMER)
d_item  = spark.table(DIM_ITEM)
d_date  = spark.table(DIM_DATE)

count_silver = silver.count()
count_fact   = fact.count()
count_cust   = d_cust.count()
count_item   = d_item.count()
count_date   = d_date.count()

print(f"silver_retail_sales : {count_silver:,}")
print(f"fact_sales          : {count_fact:,}")
print(f"dim_customer        : {count_cust:,}")
print(f"dim_item            : {count_item:,}")
print(f"dim_date            : {count_date:,}")

# COMMAND ----------
# V1 — Conteo Silver vs fact_sales

log(
    "Conteo Silver vs fact_sales",
    count_silver == count_fact,
    f"Silver={count_silver:,} | fact={count_fact:,}"
)

# COMMAND ----------
# V2 — Unicidad de transaction_id en fact_sales

distintos = fact.select("transaction_id").distinct().count()
dups      = count_fact - distintos
log("Unicidad transaction_id", dups == 0, f"duplicados={dups}")

# COMMAND ----------
# V3 — Nulos en claves primarias de dimensiones

pk_nulls = {
    "dim_customer.customer_key": d_cust.filter(F.col("customer_key").isNull()).count(),
    "dim_item.item_key":         d_item.filter(F.col("item_key").isNull()).count(),
    "dim_date.date_key":         d_date.filter(F.col("date_key").isNull()).count(),
}
for col_name, n in pk_nulls.items():
    log(f"PK nula — {col_name}", n == 0, f"nulos={n}")

# COMMAND ----------
# V4 — Nulos en foreign keys de fact_sales

fk_nulls = fact.select(
    F.sum(F.col("customer_key").isNull().cast("int")).alias("customer_key_nulls"),
    F.sum(F.col("item_key").isNull().cast("int")).alias("item_key_nulls"),
    F.sum(F.col("date_key").isNull().cast("int")).alias("date_key_nulls"),
).collect()[0]

log("FK nula — customer_key", fk_nulls["customer_key_nulls"] == 0, f"nulos={fk_nulls['customer_key_nulls']}")
log("FK nula — item_key",     fk_nulls["item_key_nulls"] == 0,     f"nulos={fk_nulls['item_key_nulls']}")
log("FK nula — date_key",     fk_nulls["date_key_nulls"] == 0,     f"nulos={fk_nulls['date_key_nulls']}")

# COMMAND ----------
# V5 — Integridad referencial fact_sales → dimensiones

huerfanos_cust = fact.join(d_cust, on="customer_key", how="left_anti").count()
huerfanos_item = fact.join(d_item, on="item_key",     how="left_anti").count()
huerfanos_date = fact.join(d_date, on="date_key",     how="left_anti").count()

log("Integridad referencial — dim_customer", huerfanos_cust == 0, f"huérfanos={huerfanos_cust}")
log("Integridad referencial — dim_item",     huerfanos_item == 0, f"huérfanos={huerfanos_item}")
log("Integridad referencial — dim_date",     huerfanos_date == 0, f"huérfanos={huerfanos_date}")

# COMMAND ----------
# V6 — Consistencia total_spent = quantity * price_per_unit

tolerancia = 0.02  # margen por redondeo decimal

inconsistentes = (
    fact
    .withColumn("total_recalc", F.round(F.col("quantity") * F.col("price_per_unit"), 2))
    .withColumn("diff", F.abs(F.col("total_spent") - F.col("total_recalc")))
    .filter(F.col("diff") > tolerancia)
    .count()
)
log("Consistencia total_spent", inconsistentes == 0, f"filas con diff>{tolerancia}={inconsistentes}")

# COMMAND ----------
# V7 — Dominios válidos: payment_method y location

invalid_payment  = fact.filter(~F.lower(F.col("payment_method")).isin(VALID_PAYMENT)).count()
invalid_location = fact.filter(~F.lower(F.col("location")).isin(VALID_LOCATION)).count()
invalid_discount = fact.filter(F.col("discount_applied").isNull()).count()

log("Dominio payment_method",  invalid_payment  == 0, f"inválidos={invalid_payment}")
log("Dominio location",        invalid_location == 0, f"inválidos={invalid_location}")
log("Dominio discount_applied (sin nulos)", invalid_discount == 0, f"nulos={invalid_discount}")

# COMMAND ----------
# V8 — Rango de fechas cubierto por dim_date

min_date_fact = fact.join(d_date, on="date_key").agg(F.min("full_date")).collect()[0][0]
max_date_fact = fact.join(d_date, on="date_key").agg(F.max("full_date")).collect()[0][0]
min_date_dim  = d_date.agg(F.min("full_date")).collect()[0][0]
max_date_dim  = d_date.agg(F.max("full_date")).collect()[0][0]

rango_ok = (min_date_dim <= min_date_fact) and (max_date_dim >= max_date_fact)
log(
    "Rango fechas dim_date cubre fact_sales",
    rango_ok,
    f"dim=[{min_date_dim},{max_date_dim}] fact=[{min_date_fact},{max_date_fact}]"
)

# COMMAND ----------
# V9 — Conteos finales de todas las tablas Gold

print("\nConteos finales Gold:")
print(f"  fact_sales   : {count_fact:,}")
print(f"  dim_customer : {count_cust:,}")
print(f"  dim_item     : {count_item:,}")
print(f"  dim_date     : {count_date:,}")

log("Conteo fact_sales > 0",   count_fact > 0,  f"filas={count_fact:,}")
log("Conteo dim_customer > 0", count_cust > 0,  f"filas={count_cust:,}")
log("Conteo dim_item > 0",     count_item > 0,  f"filas={count_item:,}")
log("Conteo dim_date = 1114",  count_date == 1114, f"filas={count_date:,}")

# COMMAND ----------
# Tabla resumen final

from pyspark.sql.types import StructType, StructField, StringType

schema_res = StructType([
    StructField("nombre_validacion", StringType(), True),
    StructField("resultado",         StringType(), True),
    StructField("detalle",           StringType(), True),
])

resumen = spark.createDataFrame(results, schema=schema_res)
display(resumen)

silver_retail_sales : 11,362
fact_sales          : 11,362
dim_customer        : 25
dim_item            : 200
dim_date            : 1,114
✓ OK — Conteo Silver vs fact_sales: Silver=11,362 | fact=11,362
✓ OK — Unicidad transaction_id: duplicados=0
✓ OK — PK nula — dim_customer.customer_key: nulos=0
✓ OK — PK nula — dim_item.item_key: nulos=0
✓ OK — PK nula — dim_date.date_key: nulos=0
✓ OK — FK nula — customer_key: nulos=0
✓ OK — FK nula — item_key: nulos=0
✓ OK — FK nula — date_key: nulos=0
✓ OK — Integridad referencial — dim_customer: huérfanos=0
✓ OK — Integridad referencial — dim_item: huérfanos=0
✓ OK — Integridad referencial — dim_date: huérfanos=0
✓ OK — Consistencia total_spent: filas con diff>0.02=0
✓ OK — Dominio payment_method: inválidos=0
✓ OK — Dominio location: inválidos=0
✓ OK — Dominio discount_applied (sin nulos): nulos=0
✓ OK — Rango fechas dim_date cubre fact_sales: dim=[2022-01-01,2025-01-18] fact=[2022-01-01,2025-01-18]

Conteos finales Gold:
  fact_sales   : 11,362


nombre_validacion,resultado,detalle
Conteo Silver vs fact_sales,✓ OK,"Silver=11,362 | fact=11,362"
Unicidad transaction_id,✓ OK,duplicados=0
PK nula — dim_customer.customer_key,✓ OK,nulos=0
PK nula — dim_item.item_key,✓ OK,nulos=0
PK nula — dim_date.date_key,✓ OK,nulos=0
FK nula — customer_key,✓ OK,nulos=0
FK nula — item_key,✓ OK,nulos=0
FK nula — date_key,✓ OK,nulos=0
Integridad referencial — dim_customer,✓ OK,huérfanos=0
Integridad referencial — dim_item,✓ OK,huérfanos=0
